Ten skrypt ładuje dane do schematu hurtowni danych

In [0]:
%sql
MERGE INTO dw.dim_car AS Dim_car
USING staging.car AS src
ON Dim_car.car_inventory_id = src.car_inventory_id
WHEN MATCHED AND Dim_car.is_active = TRUE AND (md5(CONCAT(COALESCE(Dim_car.car_producer,''),COALESCE(Dim_car.car_model,''),COALESCE(Dim_car.car_production_year,''),COALESCE(Dim_car.car_fuel_type,''),COALESCE(Dim_car.car_license_plates,''),COALESCE(Dim_car.car_current_rental_rate,0),COALESCE(Dim_car.car_purchase_price,0),COALESCE(Dim_car.car_sell_price,0))) != md5(CONCAT(COALESCE(src.car_producer,''),COALESCE(src.car_model,''),COALESCE(src.car_production_year,''),COALESCE(src.car_fuel_type,''),COALESCE(src.car_license_plates,''),COALESCE(src.car_current_rental_rate,0),COALESCE(src.car_purchase_price,0),COALESCE(src.car_sell_price,0)))) THEN 
  UPDATE SET
    is_active = false,
    valid_to = src.last_update,
    dw_update_date = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (car_inventory_id, car_id, car_producer, car_model, car_production_year, car_fuel_type, car_license_plates, car_current_rental_rate, car_purchase_price, car_sell_price, valid_from, valid_to, is_active, dw_insert_date)
VALUES(src.car_inventory_id, src.car_id, src.car_producer, src.car_model, src.car_production_year, src.car_fuel_type, src.car_license_plates, src.car_current_rental_rate, src.car_purchase_price, src.car_sell_price, src.last_update, '9999-12-31',TRUE,CURRENT_TIMESTAMP())

In [0]:
%sql
INSERT INTO dw.dim_car (car_inventory_id,car_id,car_producer,car_model,car_production_year,car_fuel_type,car_license_plates,car_current_rental_rate,car_purchase_price,car_sell_price,valid_from,valid_to,is_active,dw_insert_date)  
SELECT source.car_inventory_id,source.car_id,source.car_producer,source.car_model,source.car_production_year,source.car_fuel_type,source.car_license_plates,source.car_current_rental_rate,source.car_purchase_price,source.car_sell_price,source.last_update,'9999-12-31',TRUE,current_timestamp()
FROM staging.car source
INNER JOIN dw.dim_car target
ON source.car_inventory_id=target.car_inventory_id
and target.is_active=FALSE
AND source.last_update=target.valid_to
    


In [0]:
%sql
MERGE INTO dw.dim_customer AS dim_cust
USING (
  SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    c.email,
    c.birth_date,
    COALESCE(a.address, 'NA') as address,
    COALESCE(city.city, 'NA') as city,
    COALESCE(country.country, 'NA') as country,
    COALESCE(a.postal_code, 'NA') as postal_code,
    CASE WHEN c.last_update > a.last_update THEN c.last_update ELSE a.last_update END AS new_valid_to
  FROM landing.customer c
  INNER JOIN landing.address a ON c.address_id = a.address_id
  LEFT JOIN landing.city ON a.city_id = city.city_id
  LEFT JOIN landing.country ON city.country_id = country.country_id
) src
ON dim_cust.customer_id = src.customer_id
WHEN MATCHED AND dim_cust.is_active = TRUE AND (md5(CONCAT(COALESCE(dim_cust.customer_first_name,''),COALESCE(dim_cust.customer_last_name,''),COALESCE(dim_cust.customer_email,''),COALESCE(dim_cust.customer_birth_date,'1900-01-01'),COALESCE(dim_cust.customer_address,''),COALESCE(dim_cust.customer_city,''),COALESCE(dim_cust.customer_country,''),COALESCE(dim_cust.customer_postal_code,''))) != md5(CONCAT(COALESCE(src.first_name,''),COALESCE(src.last_name,''),COALESCE(src.email,''),COALESCE(src.birth_date,'1900-01-01'),COALESCE(src.address,''),COALESCE(src.city,''),COALESCE(src.country,''),COALESCE(src.postal_code,'')))) THEN
  UPDATE SET
    is_active = FALSE,
    valid_to = src.new_valid_to,
    dw_update_date = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (customer_id, customer_first_name, customer_last_name, customer_email, customer_birth_date, customer_address, customer_city, customer_country, customer_postal_code, valid_from, valid_to, is_active, dw_insert_date)
  VALUES (src.customer_id, src.first_name, src.last_name, src.email, src.birth_date, src.address, src.city, src.country, src.postal_code, src.new_valid_to, '9999-12-31', TRUE, current_timestamp())

In [0]:
%sql
INSERT INTO dw.dim_customer (customer_id, customer_first_name, customer_last_name, customer_email, customer_birth_date, customer_address, customer_city, customer_country, customer_postal_code, valid_from, valid_to, is_active, dw_insert_date)
SELECT 
  source.customer_id,
  source.first_name,
  source.last_name,
  source.email,
  source.birth_date,
  COALESCE(a.address, 'NA'),
  COALESCE(city.city, 'NA'),
  COALESCE(country.country, 'NA'),
  COALESCE(a.postal_code, 'NA'),
  CASE WHEN source.last_update > a.last_update THEN source.last_update ELSE a.last_update END,
  '9999-12-31',
  TRUE,
  current_timestamp()
FROM landing.customer source
INNER JOIN dw.dim_customer target ON source.customer_id = target.customer_id AND target.is_active = FALSE
INNER JOIN landing.address a ON source.address_id = a.address_id
LEFT JOIN landing.city ON a.city_id = city.city_id
LEFT JOIN landing.country ON city.country_id = country.country_id
WHERE (CASE WHEN source.last_update > a.last_update THEN source.last_update ELSE a.last_update END) = target.valid_to

In [0]:
%sql
MERGE INTO dw.dim_store d 
USING (SELECT s.store_id,s.store_manager_id,
 CONCAT(a.address,', ',a.address2) as store_address,
 coalesce(city.city,'NA') as store_city,
 coalesce(country.country,'NA') as store_country,
 coalesce(a.postal_code,'NA') as store_postal_code,
 CASE WHEN s.last_update > a.last_update THEN s.last_update 
      ELSE a.last_update END AS new_valid_to
FROM landing.store s
 INNER JOIN landing.address a ON s.address_id=a.address_id 
 INNER JOIN landing.city ON a.city_id=city.city_id  
 INNER JOIN landing.country ON city.country_id=country.country_id)
 l 
ON  l.store_id = d.store_id 
WHEN MATCHED AND d.is_active = True AND md5(CONCAT(l.store_address,l.store_city,l.store_country,l.store_postal_code,l.store_manager_id)) != md5(CONCAT(d.store_address,d.store_city,d.store_country,d.store_postal_code,d.store_manager_id)) THEN  
  UPDATE SET  
    d.is_active = False, 
    d.dw_update_date = current_timestamp(), 
    d.valid_to = new_valid_to 
WHEN NOT MATCHED THEN 
  INSERT (
    store_id,store_manager_id,store_address,store_city,store_country,store_postal_code,is_active,valid_from,valid_to,dw_insert_date)   
    VALUES(
      l.store_id,
      l.store_manager_id,
      l.store_address,
      l.store_city,
      l.store_country,
      l.store_postal_code,
      true, 
      l.new_valid_to,
      '9999-12-31',
      current_timestamp())

In [0]:
%sql
INSERT INTO dw.dim_store (
  store_id,
  store_manager_id,
  store_address,
  store_city,
  store_country,
  store_postal_code,
  is_active,
  valid_from,
  valid_to,
  dw_insert_date,
  dw_update_date
)
SELECT
  Store.store_id,
  Store.store_manager_id,
  CONCAT(a.address, ', ', a.address2) as store_address,
  coalesce(city.city,'NA') as store_city,
  coalesce(country.country,'NA') as store_country,
  coalesce(a.postal_code,'NA') as store_postal_code,
  true AS is_active,
  CASE WHEN store.last_update > a.last_update THEN store.last_update 
      ELSE a.last_update END,
  '9999-12-31' AS valid_to,
  CURRENT_TIMESTAMP() AS dw_insert_date,
  NULL AS dw_update_date
FROM landing.store Store
LEFT JOIN dw.dim_store Dim_store
  ON Dim_store.store_id = Store.store_id AND Dim_store.is_active = False
LEFT JOIN landing.address a ON Store.address_id=a.address_id
LEFT JOIN landing.city city ON a.city_id=city.city_id
LEFT JOIN landing.country country ON city.country_id=country.country_id
WHERE (CASE WHEN store.last_update > a.last_update THEN store.last_update 
      ELSE a.last_update END)=Dim_store.valid_to

In [0]:
%sql
MERGE INTO dw.dim_equipment d
USING landing.equipment l
ON d.equipment_id = l.equipment_id 
WHEN MATCHED AND md5(CONCAT(COALESCE(d.equipment_name,''),COALESCE(d.equipment_type,''),COALESCE(d.equipment_version,''))) != md5(CONCAT(COALESCE(l.name,''),COALESCE(l.type,''),COALESCE(l.version,''))) THEN UPDATE SET 
  d.equipment_name = l.name,
  d.equipment_type = l.type,
  d.equipment_version = l.version,
  d.dw_update_date = current_timestamp()
WHEN NOT MATCHED THEN INSERT (equipment_id,equipment_name,equipment_type,equipment_version,dw_insert_date)
VALUES(l.equipment_id,l.name,l.type,l.version,current_timestamp())

In [0]:
%sql

INSERT INTO dw.dim_equipment_group(equipment_group_id, equipment_group_name, is_active, dw_insert_date)
SELECT eq_group_id, eq_group_names, true, current_timestamp()
FROM staging.equipment_groups
LEFT JOIN dw.dim_equipment_group Dim_equipment_group ON Dim_equipment_group.equipment_group_id = staging.equipment_groups.eq_group_id
WHERE Dim_equipment_group.equipment_group_id IS NULL


In [0]:
%sql

INSERT INTO dw.bridge_equipment_group_equipment(equipment_group_key, equipment_key,dw_insert_date)
SELECT eq_gr.equipment_group_key,e.equipment_key,current_timestamp()
FROM dw.dim_equipment_group eq_gr
INNER JOIN staging.bridge_equipment__equipment_groups st_brg_eq_gp ON eq_gr.equipment_group_id=st_brg_eq_gp.eq_group_id
INNER JOIN dw.dim_equipment e ON st_brg_eq_gp.equipment_id=e.equipment_id
LEFT JOIN dw.bridge_equipment_group_equipment b_eg_e ON b_eg_e.equipment_group_key=eq_gr.equipment_group_key
WHERE  b_eg_e.equipment_group_key IS NULL


In [0]:
%sql
MERGE INTO dw.dim_staff d 
USING (SELECT s.staff_id,
      s.first_name,
      s.last_name,
      s.email,
      CONCAT(a.address, ', ', a.address2) as staff_address,
      coalesce(city.city,'NA') as staff_city,
      coalesce(country.country,'NA') as staff_country,
      coalesce(a.postal_code,'NA') as staff_postal_code,
      s.store_id as staff_store_id,
      s.hired_date as staff_hired_date,
      s.manager_id as staff_manager_id,
      CASE WHEN s.last_update > a.last_update THEN s.last_update 
           ELSE a.last_update END AS new_last_update
FROM landing.staff s INNER JOIN landing.address a ON s.address_id=a.address_id
LEFT JOIN landing.city city ON a.city_id=city.city_id
LEFT JOIN landing.country country ON city.country_id=country.country_id
 ) l 
ON l.staff_id = d.staff_id
WHEN MATCHED AND d.last_updated < l.new_last_update THEN  
  UPDATE SET  
    d.staff_first_name = l.first_name,
    d.staff_last_name = l.last_name,
    d.staff_email = l.email,
    d.staff_address = l.staff_address,
    d.staff_city = l.staff_city,
    d.staff_country = l.staff_country,
    d.staff_postal_code = l.staff_postal_code,
    d.staff_store_id = l.staff_store_id,
    d.staff_hired_date = l.staff_hired_date,
    d.staff_manager_id = l.staff_manager_id,
    d.dw_update_date = current_timestamp()
WHEN NOT MATCHED
THEN INSERT (staff_id, staff_first_name, staff_last_name, staff_email, staff_address, staff_city, staff_country, staff_postal_code, staff_store_id, staff_hired_date, staff_manager_id, dw_insert_date,last_updated)
VALUES (l.staff_id, l.first_name, l.last_name, l.email, l.staff_address, l.staff_city, l.staff_country, l.staff_postal_code, l.staff_store_id, l.staff_hired_date, l.staff_manager_id, current_timestamp(),new_last_update)

In [0]:
%sql
MERGE INTO dw.dim_manager d 
USING (SELECT DISTINCT m.staff_id,
      m.first_name,
      m.last_name,
      m.email,
      CONCAT(a.address, ', ', a.address2) as manager_address,
      coalesce(city.city,'NA') as manager_city,
      coalesce(country.country,'NA') as manager_country,
      coalesce(a.postal_code,'NA') as manager_postal_code,
      m.store_id as manager_store_id,
      m.hired_date as manager_hired_date,
      m.manager_id as manager_manager_id,
      CASE WHEN m.last_update > a.last_update THEN m.last_update 
           ELSE a.last_update END AS new_last_update
FROM landing.staff s INNER JOIN landing.staff m ON s.manager_id = m.staff_id
INNER JOIN landing.address a ON m.address_id=a.address_id
LEFT JOIN landing.city city ON a.city_id=city.city_id
LEFT JOIN landing.country country ON city.country_id=country.country_id
 ) l 
ON l.staff_id = d.manager_id
WHEN MATCHED AND d.last_updated < l.new_last_update THEN  
  UPDATE SET  
    d.manager_first_name = l.first_name,
    d.manager_last_name = l.last_name,
    d.manager_email = l.email,
    d.manager_address = l.manager_address,
    d.manager_city = l.manager_city,
    d.manager_country = l.manager_country,
    d.manager_postal_code = l.manager_postal_code,
    d.manager_store_id = l.manager_store_id,
    d.manager_hired_date = l.manager_hired_date,
    d.dw_update_date = current_timestamp()
WHEN NOT MATCHED
THEN INSERT (manager_id, manager_first_name, manager_last_name, manager_email, manager_address, manager_city, manager_country, manager_postal_code, manager_store_id, manager_hired_date, dw_insert_date, last_updated)
VALUES (l.staff_id, l.first_name, l.last_name, l.email, l.manager_address, l.manager_city, l.manager_country, l.manager_postal_code, l.manager_store_id, l.manager_hired_date, current_timestamp(), new_last_update)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW tmp_staff_change AS
SELECT 1 AS changed
FROM (
    SELECT staff_id, manager_id FROM landing.staff
    EXCEPT
    SELECT staff_id, staff_manager_id FROM dw.dim_staff
) diff
LIMIT 1;

In [0]:
%sql
DELETE FROM dw.bridge_staff_hierarchy
WHERE (SELECT COUNT(*) FROM tmp_staff_change) > 0;

In [0]:
%sql
WITH RECURSIVE emp_hierarchy AS (
    SELECT
        e.staff_id,
        e.manager_id,
        1 AS distance
    FROM landing.staff e
    WHERE e.manager_id IS NOT NULL
      AND e.manager_id != e.staff_id
    UNION ALL
    SELECT
        eh.staff_id,
        m.manager_id,
        eh.distance + 1 AS distance
    FROM emp_hierarchy eh
    JOIN landing.staff m
        ON eh.manager_id = m.staff_id
    WHERE m.manager_id IS NOT NULL
      AND m.manager_id != m.staff_id
),
is_empty AS (
    SELECT 1 AS empty
    FROM dw.bridge_staff_hierarchy
    LIMIT 1
)
INSERT INTO dw.bridge_staff_hierarchy(employee_id, manager_id, distance, dw_insert_date)
SELECT staff_id, manager_id, distance, current_timestamp()
FROM emp_hierarchy
WHERE
    (SELECT COUNT(*) FROM is_empty) = 0
    OR (SELECT COUNT(*) FROM tmp_staff_change) > 0;

In [0]:
%sql
INSERT INTO dw.fact_service (service_id, car_key, service_date_key, service_type, service_cost)
SELECT 
    Service.service_id,
    Dim_car.car_key,
    Dim_service_date.service_date_key,
    Service.service_type,
    Service.service_cost
FROM landing.service Service
INNER JOIN dw.dim_car Dim_car ON Dim_car.car_inventory_id = Service.inventory_id and dim_car.is_active = true
INNER JOIN dw.dim_service_date Dim_service_date ON Dim_service_date.service_date = Service.service_date
LEFT JOIN dw.fact_service Fact_service ON Fact_service.service_id = Service.service_id
WHERE Fact_service.service_id IS NULL;

In [0]:
%sql
INSERT INTO dw.fact_rental (rental_id, customer_key, car_key, store_key, staff_key, eqipment_group_key, rental_date_key, return_date, payment_date, payment_deadline, payment_delay_days, rental_amount, payment_amount, daily_rental_rate, rental_duration)
SELECT
    Rental.rental_id,
    Dim_customer.customer_key,
    Dim_car.car_key,
    Dim_store.store_key,
    Dim_staff.staff_key,
    Dim_equipment_group.equipment_group_key,
    Dim_rental_date.rental_date_key,
    Rental.return_date,
    COALESCE(Payment.payment_date, '9999-12-31') AS payment_date,
    Rental.payment_deadline,
    CASE 
        WHEN COALESCE(Payment.payment_date, '9999-12-31') < Rental.payment_deadline THEN 0
        ELSE DATEDIFF(COALESCE(Payment.payment_date, current_timestamp()), Rental.payment_deadline)
    END AS payment_delay_days,
    Rental.rental_rate * DATEDIFF(Rental.return_date, Rental.rental_date) AS rental_amount,
    COALESCE(Payment.amount,0) AS payment_amount,
    Rental.rental_rate,
    DATEDIFF(Rental.return_date, Rental.rental_date) AS rental_duration
FROM landing.rental Rental
INNER JOIN dw.dim_customer Dim_customer ON Dim_customer.customer_id = Rental.customer_id AND Dim_customer.is_active = TRUE
INNER JOIN dw.dim_car Dim_car ON Dim_car.car_inventory_id = Rental.inventory_id AND Dim_car.is_active = TRUE
INNER JOIN dw.dim_staff Dim_staff ON Dim_staff.staff_id = Rental.staff_id
INNER JOIN dw.dim_store Dim_store ON Dim_store.store_id = Dim_staff.staff_store_id AND Dim_store.is_active = TRUE
INNER JOIN staging.inventory_equipment_groups st_i_eq_g ON st_i_eq_g.inventory_id = Rental.inventory_id
INNER JOIN dw.dim_equipment_group Dim_equipment_group ON Dim_equipment_group.equipment_group_id = st_i_eq_g.eq_group_id AND Dim_equipment_group.is_active = TRUE
INNER JOIN dw.dim_rental_date Dim_rental_date ON Dim_rental_date.rental_date = Rental.rental_date
LEFT JOIN dw.fact_rental Fact_rental ON Fact_rental.rental_id = Rental.rental_id
LEFT JOIN landing.payment Payment ON Payment.rental_id = Rental.rental_id
WHERE Fact_rental.rental_id IS NULL